# Patterns that repeat {#sec-patterns}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c07-patterns.ipynb)

In chapter 6 the hospital group learnt that hiring consultants takes a year or more. While the slow fix works its way through, managers reach for quick ones: evening clinics and overtime, shorter appointments to see more patients, and treatment bought from private providers. Each brings the list down this quarter. Each can also make it worse a year or two later, through fatigue, patients who come back, or public capacity that withers because the pressure to build it has gone.

These are not accidents of one hospital. The same few feedback structures turn up again and again, in fisheries, cities, farms, payment systems and governments. System thinkers call them **system archetypes**: small loop structures with a recognisable behaviour, a typical trap and a typical way out [@senge1990; @meadows2008]. This chapter treats each archetype as a diagnostic tool for an analyst. For each one you get (a) a small model you can run, (b) a real case from a different sector or country, (c) how to recognise the pattern in data, and (d) where the leverage lies. The sketch after the setup cell shows the chapter's version of the running case.

**Who it is for.** Analysts who need to explain why a sensible policy is failing, and to say so with evidence rather than with a diagram alone. **Who can skip.** If you already use archetypes in practice, read section 1 for the four-part template, then go to section 9 (the running case) and the data signatures in each section. **Prerequisites.** Chapter 3, Loops that grow, settle and tip (the logistic and market-share molecules); chapter 5, Decisions inside the model (anchoring and goal setting); chapter 6, Why things swing (the hiring model). **Time.** About three and a half hours with the exercises.

The waiting-list data are synthetic, shaped like published Irish waiting-list counts; see chapter 10, Data, calibration and uncertainty.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package,
# which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 7
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| data:", wl.shape[0], "months,",
      wl.month.min().strftime("%b %Y"), "to", wl.month.max().strftime("%b %Y"))

In [ ]:
#| fig-alt: Stock-and-flow sketch of the chapter's case. Referrals fill the Waiting list. Patients treated well leave; those treated with flaws go up to Hidden returns and flow back to the list. Outsourced treatments also drain the list. Public capacity is built and lost. Fatigue builds from overtime. Dashed arrows show the list putting pressure on capacity building, outsourcing as a side effect draining capacity, and fatigue meaning fewer patients are treated right.
from matplotlib.patches import Circle, FancyBboxPatch


def draw_sfd(ax, stocks, flows, links):
    """A small stock-and-flow diagram (as in chapter 6)."""
    w, h = 1.7, 0.8
    for name, (x, y) in stocks.items():
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                    boxstyle="round,pad=0.02", fc="white",
                                    ec="black", lw=1.5))
        ax.text(x, y, name, ha="center", va="center", fontsize=9)

    def inside(p):
        return any(abs(p[0] - x) <= w / 2 + 1e-9
                   and abs(p[1] - y) <= h / 2 + 1e-9
                   for x, y in stocks.values())

    for label, a, b, *where in flows:
        for p in (a, b):
            if not inside(p):
                ax.text(*p, "☁", ha="center", va="center",
                        fontsize=18, color="grey")
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="-|>", lw=3, color="black",
                                    shrinkA=8, shrinkB=2))
        mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
        ax.add_patch(Circle((mx, my), 0.12, fc="white", ec="black",
                            lw=1.2, zorder=3))
        dx, dy = where[0] if where else (0, -0.3)
        ax.text(mx + dx, my + dy, label, ha="center" if not dx else
                ("left" if dx > 0 else "right"), va="top" if dy < 0 else
                "center", fontsize=8)
    for a, b, label, rad, where in links:
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", lw=1, ls="--",
                                    color="#0072B2", shrinkA=4, shrinkB=4,
                                    connectionstyle=f"arc3,rad={rad}"))
        ax.text(*where, label, ha="center", fontsize=8, color="#0072B2")
    ax.set_aspect("equal")
    ax.axis("off")


fig, ax = plt.subplots(figsize=(9, 4.6))
draw_sfd(ax,
         stocks={"Waiting list": (4, 3), "Hidden\nreturns": (4, 5.4),
                 "Public\ncapacity": (4, 0), "Fatigue": (8.5, 0.6)},
         flows=[("Referrals", (1.5, 3), (3.15, 3)),
                ("Treated and\ndischarged", (4.85, 3), (7, 3)),
                ("Outsourced", (4, 2.6), (4, 1.6), (0.25, 0)),
                ("Treated, will return", (4.4, 3.4), (4.4, 5.0), (0.25, 0)),
                ("Returning", (3.6, 5.0), (3.6, 3.4), (-0.25, 0)),
                ("Building", (1.4, 0), (3.15, 0)),
                ("Losing", (4.85, 0), (6.4, 0)),
                ("Getting fatigued", (10.6, 0.6), (9.35, 0.6))],
         links=[((4.6, 2.6), (2.2, 0.1), "pressure", 0.25, (2.4, 1.5)),
                ((4.3, 1.9), (5.6, 0.1), "side effect", -0.3, (5.9, 1.1)),
                ((8.2, 1.0), (6.1, 3.0), "fewer done right", 0.2, (7.9, 2.0))])
ax.set_xlim(0.8, 11.2)
ax.set_ylim(-0.9, 6.0)
ax.set_title("The waiting list with three quick fixes")
plt.show()

Figure: The chapter's case. Thick arrows are flows: patients above, public capacity (patients a month) below, and fatigue on the right. Dashed arrows carry information. A list above target puts pressure on managers to build capacity; outsourcing drains the list but, as a side effect, can drain public capacity; fatigue means fewer patients are treated right first time, and those come back.

**Learning outcomes.** By the end of the chapter you can:

1. **recognise** six archetypes from a model run and from the data an analyst would have (Try it 2 to 7);
2. **build** a small model of an archetype inside the running case and **test** its leverage point (Try it 2, 3, 4 and 9);
3. **estimate** a limit, a lag or an adjustment time from noisy data with least squares (sections 4, 5 and 8);
4. **diagnose** when a quick fix turns into a dependency, by finding the threshold at which it does (Try it 4);
5. **justify** a combined policy to a decision maker with a scorecard of costs and benefits (section 9 and the mini-project).

## 1. Archetypes as diagnostic patterns

An archetype is a small set of loops that, wherever it appears, produces the same kind of behaviour over time. It helps the analyst in two ways. Faced with a puzzling trend, it suggests a structure to look for; faced with a structure, it says what the trend will look like and where an intervention will pay off. It is a hypothesis to test, not a conclusion: the model and the data decide.

| Archetype | Loops | Behaviour you see | Typical leverage |
|---|---|---|---|
| Fixes that fail | a balancing fix and a delayed side effect that undoes it | improvement, then a problem as bad or worse | find and weaken the side effect; use the fix sparingly |
| Shifting the burden | a quick symptomatic fix and a slow fundamental one; the quick fix weakens the slow one | dependence on the quick fix grows; capability falls | protect the fundamental fix; time-limit the quick one |
| Limits to growth | a reinforcing engine meets a balancing limit | S-shaped growth that stalls | raise or remove the limit, not the engine |
| Tragedy of the commons | several users each grow on a shared resource that depletes | boom, then collapse for everyone | manage the commons as a whole (limits, shares) |
| Success to the successful | two parties compete for a resource that goes to whoever already has more | small early differences become dominance | weaken the link between success and allocation |
| Eroding goals | a gap can be closed by improving or by lowering the goal | standards drift down with performance | anchor the goal to something outside the system |

We will run each on a model, then tell a real case in a few lines, then show what the archetype leaves behind in the kind of data an analyst actually gets. We start by building the running case once, because three of the archetypes live inside it.

### The running case for this chapter

The model extends the waiting list of chapter 1 with the quick fixes, one switch for each, all off by default:

- **Overtime** (`Overtime_limit`, 1 means none): when the work wanted exceeds public capacity, staff work up to this multiple of normal hours. Hours build **fatigue**, a smooth of overtime over 3 months, which cuts productivity and the share of patients treated right first time.
- **Rushed clinics** (`Rush_limit`, 1 means none): appointments are shortened to see more patients, at a cost in quality (`Rush_quality_cost`).
- **Outsourcing** (`Outsourcing_share`, a fraction): a share of the excess list is sent to private providers each month. Outsourced activity is counted when planning public capacity, and each outsourced patient a month drains `Staff_drift` patients a month of public capacity each month (staff sessions and budget move to private work).
- **Eroding goals** (`Goal_protection`, 1 means the target is fixed): with less protection, the target list drifts towards the actual list over 24 months.

Patients treated with flaws go to `Hidden returns` and come back to the list after 4 months on average. Public capacity is rebuilt as it wears out (a life of 120 months) and grows towards the treatments wanted over 18 months. Referrals rise by 10% in month 12, as in chapter 6.

In [ ]:
case_spec = {
    "name": "Waiting list with quick fixes", "timeUnit": "month",
    "T": 120, "dt": 0.125,
    "stocks": [
        {"id": "wl", "name": "Waiting list", "init": "32000",
         "units": "patients", "nonNegative": True},
        {"id": "rw", "name": "Hidden returns", "init": "0",
         "units": "patients", "nonNegative": True},
        {"id": "cap", "name": "Public capacity", "init": "3950",
         "units": "patients/month", "nonNegative": True},
        {"id": "fat", "name": "Fatigue", "init": "1",
         "units": "dimensionless"},
        {"id": "tgt", "name": "Target list", "init": "32000",
         "units": "patients"}],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl",
         "eq": "4000 + STEP(Referral_rise, 12)", "units": "patients/month"},
        {"id": "ok", "name": "Treated and discharged", "from": "wl",
         "eq": "Public_treatments * Quality", "units": "patients/month"},
        {"id": "bad", "name": "Treated but will return", "from": "wl",
         "to": "rw", "eq": "Public_treatments * (1 - Quality)",
         "units": "patients/month"},
        {"id": "ret", "name": "Returning", "from": "rw", "to": "wl",
         "eq": "Hidden_returns / Time_to_return", "units": "patients/month"},
        {"id": "out", "name": "Outsourced treatments", "from": "wl",
         "units": "patients/month",
         "eq": "Outsourcing_share * (1 - STEP(1, Outsourcing_end))"
               " * Excess_list / Time_to_reduce_list"},
        {"id": "inv", "name": "Building capacity", "to": "cap",
         "units": "patients/month/month",
         "eq": "MAX(0, Public_capacity / Capacity_life + (Desired_treatments"
               " - Public_capacity - Outsourced_treatments)"
               " / Time_to_build_capacity)"},
        {"id": "loss", "name": "Losing capacity", "from": "cap",
         "units": "patients/month/month",
         "eq": "Public_capacity / Capacity_life"
               " + Staff_drift * Outsourced_treatments"},
        {"id": "fg", "name": "Getting fatigued", "to": "fat",
         "units": "1/month", "eq": "(Overtime - Fatigue) / Time_to_fatigue"},
        {"id": "ge", "name": "Goal erosion", "to": "tgt",
         "units": "patients/month",
         "eq": "(1 - Goal_protection) * (Waiting_list - Target_list)"
               " / Goal_adjustment_time"}],
    "vars": [
        {"id": "rise", "name": "Referral rise", "eq": "400",
         "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"},
        {"id": "tred", "name": "Time to reduce list", "eq": "12",
         "units": "month"},
        {"id": "ex", "name": "Excess list", "units": "patients",
         "eq": "MAX(0, Waiting_list - Target_list)"},
        {"id": "dtr", "name": "Desired treatments", "units": "patients/month",
         "eq": "Referrals + (Waiting_list - Target_list)"
               " / Time_to_reduce_list"},
        {"id": "sp", "name": "Schedule pressure", "units": "dimensionless",
         "eq": "MAX(0, Desired_treatments - Outsourced_treatments)"
               " / Public_capacity"},
        {"id": "otl", "name": "Overtime limit", "eq": "1",
         "units": "dimensionless"},
        {"id": "ot", "name": "Overtime", "units": "dimensionless",
         "eq": "MAX(1, MIN(Overtime_limit, Schedule_pressure))"},
        {"id": "ttf", "name": "Time to fatigue", "eq": "3", "units": "month"},
        {"id": "fs", "name": "Fatigue strength", "eq": "0.8",
         "units": "dimensionless"},
        {"id": "ef", "name": "Effect of fatigue on productivity",
         "units": "dimensionless",
         "eq": "MAX(0.3, MIN(1.1, 1 - Fatigue_strength * (Fatigue - 1)))"},
        {"id": "rl", "name": "Rush limit", "eq": "1",
         "units": "dimensionless"},
        {"id": "rush", "name": "Rush", "units": "dimensionless",
         "eq": "MAX(1, MIN(Rush_limit, Schedule_pressure / Overtime))"},
        {"id": "pt", "name": "Public treatments", "units": "patients/month",
         "eq": "MIN(Public_capacity * Overtime * Rush"
               " * Effect_of_fatigue_on_productivity,"
               " Waiting_list / Minimum_treatment_time)"},
        {"id": "nq", "name": "Normal quality", "eq": "1", "units": "fraction"},
        {"id": "rqc", "name": "Rush quality cost", "eq": "1",
         "units": "dimensionless"},
        {"id": "q", "name": "Quality", "units": "fraction",
         "eq": "MAX(0, MIN(1, Normal_quality"
               " * MAX(0.4, 1 - 0.4 * (Fatigue - 1))"
               " * MAX(0.4, 1 - Rush_quality_cost * (Rush - 1))))"},
        {"id": "ttr", "name": "Time to return", "eq": "4", "units": "month"},
        {"id": "os", "name": "Outsourcing share", "eq": "0",
         "units": "fraction"},
        {"id": "oe", "name": "Outsourcing end", "eq": "1000",
         "units": "month"},
        {"id": "cl", "name": "Capacity life", "eq": "120", "units": "month"},
        {"id": "tbc", "name": "Time to build capacity", "eq": "18",
         "units": "month"},
        {"id": "sd", "name": "Staff drift", "eq": "0.02", "units": "1/month"},
        {"id": "gp", "name": "Goal protection", "eq": "1",
         "units": "fraction"},
        {"id": "gat", "name": "Goal adjustment time", "eq": "24",
         "units": "month"}]}
case = lb.Model.from_spec(case_spec)
print(case.equations())


def month_axis(ax):
    """Label a month axis with the running case's calendar years."""
    ticks = np.arange(0, 121, 24)
    ax.set_xticks(ticks, [str(2014 + t // 12) for t in ticks])
    ax.set_xlabel("year")


base = case.run()
print(f"\nbase: list peaks at {base['Waiting list'].max():,.0f} in month "
      f"{base['Waiting list'].idxmax():.0f}; month 120: "
      f"{base['Waiting list'].iloc[-1]:,.0f} patients, capacity "
      f"{base['Public capacity'].iloc[-1]:,.0f} patients/month")

With no quick fixes, the list rises after the referral rise to 35,869 patients in month 30, while capacity is built, and is back at 32,169 by month 120 with capacity at 4,386 patients a month. This is the run every fix will be compared with.

## 2. Fixes that fail: overtime and fatigue

**The pattern.** A problem prompts a fix that works quickly. The fix also starts a slower process that brings the problem back, often worse. Because the improvement comes first and the damage later, the fix looks good to the people who chose it, and when the problem returns they apply more of it.

**(a) The molecules.** Three molecules give the pieces. **Overtime** (`overtime`), after Hines, *Molecules of Structure* [@hines_molecules], turns a shortfall of people into hours: the overtime wanted is the people needed divided by the people available, and a limit caps what people will actually work,

$$
\text{Overtime} = \min\!\Big(\text{limit},\ \frac{R^{*}/p}{W}\Big).
$$

**Effect of Fatigue** (`effectFatigue`), after Hines, *Molecules of Structure* [@hines_molecules], makes fatigue $F$ a smooth of overtime with a time to get fatigued $T_F$, in the same units as overtime, and turns it into an effect on productivity:

$$
\frac{dF}{dt} = \frac{\text{Overtime} - F}{T_F}, \qquad \text{Effect} = \min\!\big(1.2,\ \max(0,\ 1 - s\,(F - 1))\big).
$$

**Productivity, Overtime and Fatigue** (`prodFatigue`), after Hines, *Molecules of Structure* [@hines_molecules], puts them together for a team of 20, with a strong effect ($s = 1$, written $2 - F$). Our versions use straight-line effects with caps where Hines draws lookup curves; the case model uses $s = 0.8$.

In [ ]:
#| fig-alt: Output of a team over 60 weeks at normal hours, 20% and 40% overtime. Output jumps with overtime, then falls as fatigue builds; at 40% overtime it ends below normal.
ot = lb.molecule("overtime")
ef = lb.molecule("effectFatigue")
pf = lb.molecule("prodFatigue")
for workers in (10, 12, 15):
    d = ot.run(T=20, workers=workers)
    print(f"Overtime, {workers} workers: overtime "
          f"{d['Overtime'].iloc[-1]:.2f}, behind after 20 weeks "
          f"{d['Work fallen behind'].iloc[-1]:.0f} tasks")
for label in ("Sustained crunch", "Slow to tire", "Short weeks (rested)"):
    d = ef.run(preset=label)
    print(f"Effect of Fatigue, {label:<20}: effect on productivity month 6 "
          f"{d.loc[6.0, 'Effect on PDY']:.2f}, month 60 "
          f"{d['Effect on PDY'].iloc[-1]:.2f}")

fig, ax = plt.subplots()
for o in (1, 1.2, 1.4):
    out = pf.run(overtime=o)["Output"]
    out.plot(ax=ax, label=f"overtime {o}")
    print(f"Productivity and fatigue, overtime {o}: week 0 {out.iloc[0]:.0f},"
          f" week 60 {out.iloc[-1]:.0f}, total "
          f"{out.iloc[:-1].sum() * 0.1:,.0f} tasks")
ax.set(title="Overtime and fatigue: output of a team of 20",
       xlabel="time (week)", ylabel="output (tasks/week)")
ax.legend(fontsize=8)
plt.show()

Figure: Output of a team of 20 (normal productivity 5 tasks a person a week) working normal hours, 20% and 40% overtime, in the Productivity, Overtime and Fatigue molecule.

With 10 workers and work for 15, the Overtime molecule hits its limit of 1.3 and falls 160 tasks behind in 20 weeks; with 12 workers, 25% overtime covers the gap. Effect of Fatigue shows how the cost arrives: in a sustained crunch of 50% overtime, productivity has fallen to 0.74 of normal by month 6 and 0.70 by month 60, and a slower-tiring team gets to the same place later. In the third molecule, 40% overtime lifts output to 140 tasks a week at once, but by week 60 it is 84, below the 100 of normal hours, and over 60 weeks the team delivers 5,264 tasks against 6,000. Every gain comes first and every loss later.

Now the running case: clinics work up to 30% overtime whenever the work wanted exceeds public capacity.

In [ ]:
#| fig-alt: Difference between the waiting list with overtime allowed and without, over 120 months. It is negative for about two years, then positive, peaking near 2,400 patients in 2018, then negative again after 2020.
ovt = case.run(overtime_limit=1.3)
diff = ovt["Waiting list"] - base["Waiting list"]
fig, ax = plt.subplots()
diff.plot(ax=ax)
ax.axhline(0, color="black", lw=0.8)
ax.set(title="Overtime: list with overtime minus list without",
       ylabel="difference (patients)")
month_axis(ax)
plt.show()
worse = diff[(diff.index > 12) & (diff > 0)]
print(f"overtime helps until month {worse.index.min():.1f} (best "
      f"{diff.min():,.0f} patients); hurts most in month "
      f"{diff.idxmax():.0f} (+{diff.max():,.0f})")
print(f"fatigue peaks at {ovt['Fatigue'].max():.2f}, quality falls to "
      f"{ovt['Quality'].min():.3f}, returns reach "
      f"{ovt['Returning'].max():.0f} patients/month")

# What an analyst would see: the fix first, its side effects later
for col in ("Overtime", "Fatigue", "Returning"):
    print(f"{col:<10} peaks in month {ovt[col].idxmax():5.1f}")
print(f"list gap (with minus without) peaks in month {diff.idxmax():.1f}")

Figure: The waiting list with overtime allowed (limit 1.3) minus the list without it. Below zero, overtime is helping.

Overtime helps at first: the list is up to 859 patients shorter than without it. From month 26.4 it is longer, by as much as 2,416 patients in month 50. Fatigue rises to 1.10 (10% above normal), so productivity falls and the share treated right first time falls to 96.1%; up to 162 patients a month come back. Worse, while overtime is holding the list down, managers see less need to build capacity, so the extra capacity arrives later. The fix fails twice: through fatigue and through the delayed investment. Later still the difference turns negative again, because the bigger list eventually forced more capacity to be built: in an archetype, the timing is the story.

**(b) A real case: driving bans in Mexico City.** In 1989 Mexico City restricted each car from driving on one weekday, chosen by the last digit of its number plate, to cut air pollution. Traffic fell at first. But many households responded by buying a second car, often an older and dirtier one, so they could still drive every day, and later studies found little or no lasting improvement in air quality. The fix (fewer cars on the road) set off a slow side effect (more, and worse, cars) that undid it.

**(c) How an analyst recognises it.** Look for a sequence in time: the fix rises first, then its side effects, then the problem. In the run above, overtime jumps in month 12.0, fatigue peaks in month 23.5, returns in month 29.9, and the damage to the list in month 50.5. In real records the analyst would look for the same order in overtime hours, sick leave or staff turnover, and re-attendance rates, and would compare the list with a counterfactual (a model run, or a similar unit that did not use the fix) rather than with last quarter. A before-and-after comparison over a few months will always flatter the fix.

**(d) Leverage.** Weaken the side effect or use the fix only briefly. The exercise shows how much depends on the strength of fatigue.

**Try it 2.**

- *Check.* In Productivity, Overtime and Fatigue, what is long-run output at 20% overtime? Derive it before running.
- *Apply.* In the case, limit overtime to 1.05, 1.1 and 1.3. How does the list in month 36 compare with no overtime?
- *Extend.* With 30% overtime allowed, set the fatigue strength to 0.8, 0.4 and 0. When does overtime stop backfiring?

<details><summary>Worked answer</summary>

Fatigue settles at the overtime level, 1.2, so the effect is 2 minus 1.2 = 0.8 and output is 100 times 1.2 times 0.8 = 96 tasks a week; the run gives 96.0. In the case, the list in month 36 is 36,061 with a limit of 1.05, 36,929 with 1.1 and 36,957 with 1.3, against 35,546 with no overtime: even a little overtime is worse by then (the pressure in the case rarely calls for more than about 10%, so 1.1 and 1.3 are almost the same). The peak of the list is 36,995 with a fatigue strength of 0.8, 34,615 with 0.4 and 33,079 with 0, against 35,869 without overtime. With a weak fatigue effect overtime genuinely helps. Whether this archetype applies is an empirical question about the strength of the side effect, which is why the analyst has to measure it.
</details>

In [ ]:
# Check: long-run output of prodFatigue at overtime 1.2
print("overtime 1.2, week 60:",
      round(pf.run(overtime=1.2)["Output"].iloc[-1], 1))
# Apply: a gentler overtime limit in the case
for limit in (1.05, 1.1, 1.3):
    d = case.run(overtime_limit=limit)
    print(f"limit {limit}: list in month 36 {d.loc[36.0, 'Waiting list']:,.0f}"
          f" (no overtime {base.loc[36.0, 'Waiting list']:,.0f})")
# Extend: with a weaker fatigue effect, does overtime still backfire?
for s in (0.8, 0.4, 0.0):
    d = case.run(overtime_limit=1.3, fatigue_strength=s)
    print(f"fatigue strength {s}: list peak {d['Waiting list'].max():,.0f}"
          f" (no overtime {base['Waiting list'].max():,.0f})")

## 3. Rushed clinics: schedule pressure, quality and rework

Rushing is the other quick way to see more patients: shorter appointments, less time for tests and explanations. It is a second fix that can fail, through **rework**: work that looked done but has to be done again.

**(a) The molecules.** **Scheduled Completion and Pressure** (`schedulePressure`), after Hines, *Molecules of Structure* [@hines_molecules], sets the pace from the work remaining $R$ and the time left before a deadline: schedule pressure is the required rate over the normal rate, and the work rate rises with it up to a cap,

$$
\text{Pressure} = \frac{R / \max(\text{Deadline} - t,\ 0.25)}{\text{Normal rate}}, \qquad \text{Work rate} = \text{Normal rate} \times \min(2, \max(0.5, \text{Pressure})).
$$

**Quality** (`quality`), after Hines, *Molecules of Structure* [@hines_molecules], is the fraction of work done correctly: a normal quality multiplied by effects of fatigue, schedule pressure, work adequacy and skill, capped at 1. **Rework Cycle** (`reworkCycle`), after Hines, *Molecules of Structure* [@hines_molecules], sends the flawed fraction $1 - q$ of all work into a stock of **undiscovered rework** that flows back to the work to do after a discovery time $T_D$:

$$
\text{Correct work} = A\,q, \qquad \text{Creating rework} = A\,(1 - q), \qquad \text{Discovering rework} = \frac{U}{T_D}.
$$

Our versions use straight-line effects with caps in place of Hines' lookup functions.

In [ ]:
#| fig-alt: Two panels. Left, work done in the Rework Cycle at quality 0.95 and 0.6 over 60 weeks; low quality takes much longer to finish. Right, work remaining against a 15-week and an 8-week deadline under schedule pressure.
sch = lb.molecule("schedulePressure")
qm = lb.molecule("quality")
rc = lb.molecule("reworkCycle")
for label in ("Normal conditions", "Deadline crunch", "Green team"):
    q = qm.run(T=0, preset=label)["Quality"].iloc[0]
    print(f"Quality, {label:<17}: quality {q:.3f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for label in ("High quality", "Low quality (long tail)"):
    d = rc.run(preset=label)
    d["Work done"].plot(ax=a1, label=label)
    done = d["Work done"]
    t90, t99 = done.index[done >= 900][0], done.index[done >= 990][0]
    print(f"Rework Cycle, {label:<24}: 90% done in week {t90:.1f}, "
          f"99% in week {t99:.1f}")
for label in ("On schedule", "Tight deadline (slips)"):
    d = sch.run(T=30, preset=label)
    d["Work remaining"].plot(ax=a2, label=label)
    left = d["Work remaining"]
    print(f"Schedule pressure, {label:<22}: peak pressure "
          f"{d['Schedule pressure'].max():.2f}, done (under 1 task) in week "
          f"{left.index[left < 1][0]:.1f}")
a1.set(title="Rework Cycle", xlabel="time (week)", ylabel="work done (tasks)")
a2.set(title="Schedule pressure", xlabel="time (week)",
       ylabel="work remaining (tasks)")
a1.legend(fontsize=8)
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Left, Rework Cycle: work done at quality 0.95 and 0.6 (capacity 90 tasks a week, discovery time 5 weeks). Right, Schedule pressure: work remaining with a 15-week and an 8-week deadline.

Normal conditions give a quality of 0.900; a deadline crunch (fatigue 1.5, pressure 2) gives 0.504, and an inexperienced team 0.576. In the Rework Cycle, a high-quality project is 90% done in week 10.6 and 99% done in week 15.8. At a quality of 0.6 it is 90% done in week 20.6 but needs until week 40.5 for the last 9%: the familiar project that is "90% done" for half its life. In Schedule pressure, the 8-week deadline is met in week 8.3 only by doubling the pace, which is the molecule's cap; the case model links that kind of pressure to quality.

In the running case, clinics may speed up by up to 30% when public capacity falls short, with quality falling one for one with the speed-up.

In [ ]:
#| fig-alt: Two panels over 120 months. Left, the waiting list with and without rushed clinics. Right, patients returning each month; with rushing, returns rise to several hundred a month after the referral rise.
rush = case.run(rush_limit=1.3)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for label, d in (("no rushing", base), ("rushed clinics", rush)):
    d["Waiting list"].plot(ax=a1, label=label)
    d["Returning"].plot(ax=a2, label=label)
a1.set(title="Waiting list", ylabel="patients")
a2.set(title="Patients returning", ylabel="patients/month")
for a in (a1, a2):
    month_axis(a)
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
for m in (24, 36, 60):
    print(f"month {m}: list {base.loc[float(m), 'Waiting list']:,.0f} without,"
          f" {rush.loc[float(m), 'Waiting list']:,.0f} with rushing")
share = rush["Returning"] / rush["Referrals"]
print(f"quality falls to {rush['Quality'].min():.3f}; returns peak at "
      f"{share.max():.1%} of referrals in month {share.idxmax():.0f}")

Figure: The running case with and without rushed clinics. Left, the waiting list; right, patients coming back to the list each month.

Rushing wins for about two years: in month 24 the list is 34,689 against 35,560. By month 36 it is behind (35,810 against 35,546), and in month 60 it is 33,370 against 32,169. Quality falls to 0.889, and in month 24 the patients coming back are equal to 9.3% of new referrals. Those returns are invisible when the decision to rush is taken; they surface months later as "new" demand.

**(b) A real case: hospital readmissions in the United States.** By the late 2000s, about one in five older patients discharged from American hospitals was back in hospital within a month, and many of those returns were linked to rushed discharges and poor follow-up. Shorter stays had looked like efficiency. From 2012 the Medicare programme began to fine hospitals with high readmission rates for some conditions, putting a price on the rework that shorter stays had hidden.

**(c) How an analyst recognises it.** Look for demand that is not new: re-referrals, return visits, complaints or corrections, as a share of all activity, rising some months after a push on throughput. In project data, the signature is progress that slows near the end (the long tail above) and estimates of "percent complete" that stall. If the data system cannot tell a returning patient from a new one, that is the first thing to fix.

**(d) Leverage.** Find rework early (a shorter discovery time) and protect quality under pressure. The exercise tests both.

**Try it 3.**

- *Check.* Work out the Deadline crunch quality by hand: normal quality 0.9, fatigue 1.5 with an effect of $1 - 0.4(F - 1)$, schedule pressure 2 with an effect of $1 - 0.3(P - 1)$.
- *Apply.* In the Low quality preset of Rework Cycle, set the discovery time to 2, 5 and 10 weeks. When is the project 99% done?
- *Extend.* In the case, halve and quarter `Rush_quality_cost`. How much of the damage in month 60 goes away?

<details><summary>Worked answer</summary>

0.9 times 0.8 times 0.7 = 0.504, as the molecule says. The project is 99% done in week 25.0 with a 2-week discovery time, 40.5 with 5 and 69.9 with 10: finding flaws sooner is worth more than almost anything else in a rework cycle. In the case, the list in month 60 is 33,370 with the full quality cost, 32,552 at half and 32,257 at a quarter, against 32,169 with no rushing. Most of the damage of rushing is the rework, not the rushing itself.
</details>

In [ ]:
# Check: quality in the Deadline crunch preset, by hand
f, s = 1 - 0.4 * (1.5 - 1), 1 - 0.3 * (2 - 1)
print("by hand:", round(0.9 * f * s, 3))
# Apply: a faster discovery of rework in the Rework Cycle
for t in (5, 2, 10):
    d = rc.run(T=100, preset="Low quality (long tail)",
               rework_discovery_time=t)
    done = d["Work done"]
    print(f"discovery time {t:>2}: 99% done in week "
          f"{done.index[done >= 990][0]:.1f}")
# Extend: rushing that costs less quality
for c in (1, 0.5, 0.25):
    d = case.run(rush_limit=1.3, rush_quality_cost=c)
    print(f"quality cost {c}: list in month 60 "
          f"{d.loc[60.0, 'Waiting list']:,.0f}")

## 4. Shifting the burden: outsourcing

**The pattern.** A problem can be met in two ways: a quick, symptomatic fix, or a slower fundamental one. The quick fix relieves the symptom, which takes away the pressure for the fundamental one, and often it also has a side effect that makes the fundamental one harder. Over time the system comes to depend on the quick fix.

**(a) The molecule.** **Shifting the Burden** (`shiftingBurden`) is our own model of the archetype Senge describes [@senge1990], not one of Hines' molecules, though it is built from them (a split flow and stocks that drain at a residence time). It has a team with open problems $P$. A share $s$ of them is patched quickly; patches fail after a patch life and the problems return. The rest are fixed fundamentally at a rate that depends on the team's **capability** $C$, which is built only with spare time and fades otherwise:

$$
\text{Quick fixes} = \min(s P,\ \text{capacity}), \qquad \text{Fundamental fixes} = \frac{(1 - s)\,P\,C/100}{2}, \qquad \frac{dC}{dt} = \frac{\text{spare} \times (100 - C)}{T_B} - \frac{C}{25}.
$$

Our version is a compact composition for this archetype: patches, fundamental fixes and capability are written as simple first-order flows, with no lookup curves.

In [ ]:
#| fig-alt: "Two panels over 60 weeks for three policies in Shifting the Burden. Open problems: firefighting falls first, then climbs far above the others. Capability: it collapses under firefighting and holds up when the team invests in the fix."
sb = lb.molecule("shiftingBurden")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for label in ("Invest in the fix", "Balanced", "Firefighting"):
    d = sb.run(preset=label)
    d["Open problems"].plot(ax=a1, label=label)
    d["Capability"].plot(ax=a2, label=label)
    print(f"{label:<17}: open problems low {d['Open problems'].min():5.1f} "
          f"(week {d['Open problems'].idxmin():4.1f}), week 60 "
          f"{d['Open problems'].iloc[-1]:6.1f}; capability week 60 "
          f"{d['Capability'].iloc[-1]:5.1f}")
a1.set(title="Open problems", xlabel="time (week)", ylabel="problems")
a2.set(title="Capability", xlabel="time (week)", ylabel="capability (index)")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Shifting the Burden for three policies: patch 20%, 50% or 80% of problems quickly.

Firefighting looks best at first: open problems fall from 73 to 59.2 by week 2.9. Then patches fail, capability decays because there is no spare time to build it (to 12.6 by week 60), and open problems climb to 239.7. Investing in the fix looks worse at first and ends with 34.6 open problems and capability 65.4.

In the running case the quick fix is outsourcing: 90% of the excess list each month is sent to private providers. The fundamental fix is public capacity. Two links make this an archetype rather than a simple top-up: capacity planning counts outsourced activity as if it were permanent, and outsourcing drains public capacity through `Staff_drift`. We do not know how strong that drain is, so we test a range.

In [ ]:
#| fig-alt: "Two panels over 120 months for staff drift of 0, 0.05, 0.07 and 0.1 with 90% of the excess list outsourced. Left, public capacity: it falls further the stronger the drift. Right, outsourced treatments: they fade at low drift and grow without end at 0.1."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
rows = {}
for drift in (0, 0.05, 0.07, 0.1):
    d = case.run(outsourcing_share=0.9, staff_drift=drift)
    d["Public capacity"].plot(ax=a1, label=f"drift {drift}")
    d["Outsourced treatments"].plot(ax=a2, label=f"drift {drift}")
    rows[drift] = {"list peak": d["Waiting list"].max(),
                   "list month 120": d["Waiting list"].iloc[-1],
                   "capacity month 120": d["Public capacity"].iloc[-1],
                   "outsourced month 120": d["Outsourced treatments"].iloc[-1]}
a1.set(title="Public capacity", ylabel="patients/month")
a2.set(title="Outsourced treatments", ylabel="patients/month")
for a in (a1, a2):
    month_axis(a)
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(pd.DataFrame(rows).T.round(0).astype(int).to_string())
print(f"\nno outsourcing: list peak {base['Waiting list'].max():,.0f}")
print("threshold 1 / (share x build time) =",
      round(1 / (0.9 * 18), 3), "per month")

Figure: The running case with 90% of the excess list outsourced, for four strengths of staff drift (patients a month of public capacity lost each month per outsourced patient a month).

With no drift, outsourcing is a bridge: the list peaks at 34,457 instead of 35,869, outsourcing fades to almost nothing, and capacity reaches 4,400. At a drift of 0.05 the bridge becomes a crutch: 133 patients a month are still outsourced in month 120 and capacity is 120 patients a month lower. At 0.07 outsourcing grows every year (399 a month by month 120) and at 0.1 public capacity falls to 2,359, outsourcing reaches 1,626 a month and the list climbs to 53,676. There is a **threshold**: the burden shifts for good when the drain per outsourced patient beats the rate at which planning rebuilds capacity, about 1 / (0.9 × 18 months) = 0.062 per month. Below it, outsourcing helps; above it, it hollows out the public system. Whether a real health system is above or below that line is the question an analyst should put to the data.

**(b) A real case: pesticides and rice in Indonesia.** In the 1970s and early 1980s Indonesia subsidised heavy pesticide use to protect its rice crop. The sprays killed the brown planthopper, but they also killed the spiders and insects that ate it, so outbreaks returned larger and farmers sprayed more. In 1986 the government banned dozens of rice pesticides, cut the subsidies and trained farmers in integrated pest management, rebuilding the natural control that the quick fix had destroyed. Health systems that buy activity from private providers, such as Ireland's National Treatment Purchase Fund since 2002, face the same question of whether purchased care is a bridge or a substitute for public capacity.

**(c) How an analyst recognises it.** The share of the problem handled by the quick fix trends up while an indicator of the fundamental capability (staffed posts, theatre sessions, skills) trends down, and the system gets worse whenever the quick fix is interrupted. The code below simulates what an analyst would see in quarterly returns from the drift-0.1 run, fits a straight-line trend with least squares, and then tests two leverage points.

In [ ]:
# What an analyst would see: the outsourced share of all treatments,
# observed quarterly with noise, and its trend fitted by least squares
d = case.run(outsourcing_share=0.9, staff_drift=0.1)
q = d.iloc[::24]                                  # one row a quarter
share = (q["Outsourced treatments"]
         / (q["Outsourced treatments"] + q["Public treatments"]))
obs = share * rng.lognormal(0, 0.05, len(share))
late = obs[obs.index >= 36]
slope, _ = np.polyfit(late.index, late.values, 1)
print(f"outsourced share: {obs.iloc[4]:.1%} in month 12+, "
      f"{obs.iloc[-1]:.1%} in month 120; trend from month 36 "
      f"{slope * 12:+.1%} points a year")

# Leverage: plan public capacity as if outsourcing will stop
plan = case.with_equation(
    "Building capacity",
    "MAX(0, Public_capacity / Capacity_life + (Desired_treatments"
    " - Public_capacity) / Time_to_build_capacity)")
for label, m, kw in (("plan counts outsourcing", case, {}),
                     ("plan ignores outsourcing", plan, {}),
                     ("3-year sunset on outsourcing", case,
                      {"outsourcing_end": 36})):
    d = m.run(outsourcing_share=0.9, staff_drift=0.1, **kw)
    print(f"{label:<29}: list peak {d['Waiting list'].max():7,.0f}, "
          f"month 120 {d['Waiting list'].iloc[-1]:7,.0f}, capacity "
          f"{d['Public capacity'].iloc[-1]:5,.0f}")

The outsourced share of all treatments rises from 0.6% in month 12 to 40.3% in month 120, a trend of about 4.5 percentage points a year from month 36: a clear signal, even with noise, long before capacity collapses.

**(d) Leverage.** Two changes each break the trap at a drift of 0.1. If capacity is planned as if outsourcing will stop (the plan ignores outsourced activity), the list peaks at 34,947 and capacity ends at 4,319. A three-year sunset on outsourcing gives a higher peak (38,449), because the support stops while capacity is still being built, but the list ends at 31,479 with capacity 4,337. Both protect the fundamental fix; the sunset also limits the side effect.

**Try it 4.**

- *Check.* In Shifting the Burden, run 200 weeks with the share patched quickly at 0.1, 0.3, 0.5, 0.6 and 0.8. Is the damage gradual?
- *Apply.* The threshold formula says the drift that tips the case is 1 / (share × build time). Halve the build time to 9 months with a drift of 0.07. Does outsourcing still grow?

<details><summary>Worked answer</summary>

Up to 0.5 more patching costs a little capability each time (66.8 at 0.1, 64.1 at 0.3, 53.1 at 0.5). At 0.6 quick fixes use the whole team capacity, spare time falls to zero, capability collapses to 0.5 and open problems pass 1,000 (1,096.7). The damage is not gradual: there is a tipping point between 0.5 and 0.6. With a build time of 9 months the threshold doubles to about 0.12, so a drift of 0.07 is now below it: outsourcing in month 120 falls from 399 to 16 patients a month. Building capacity faster makes the system much harder to tip.
</details>

In [ ]:
# Check: Shifting the Burden, share patched quickly from 0.1 to 0.8
for share in (0.1, 0.3, 0.5, 0.6, 0.8):
    d = sb.run(T=200, share_patched_quickly=share)
    print(f"share {share}: open problems {d['Open problems'].iloc[-1]:7.1f},"
          f" capability {d['Capability'].iloc[-1]:5.1f}")
# Apply: a faster build time moves the threshold
for tbc in (18, 9):
    d = case.run(outsourcing_share=0.9, staff_drift=0.07,
                 time_to_build_capacity=tbc)
    print(f"build time {tbc}: outsourced in month 120 "
          f"{d['Outsourced treatments'].iloc[-1]:.0f} patients/month")

## 5. Limits to growth

**The pattern.** A reinforcing loop drives growth, and the growth itself uses up something it needs: space, water, customers, staff. Growth slows and stalls. Pushing harder on the engine does little; the leverage is in the limit.

**(a) The model.** The Constrained Growth molecule (`logistic`) from chapter 3 is the archetype in its simplest form: growth $rP$ and crowding $rP^2/K$ for a carrying capacity $K$. The first lines below compare a faster engine with a higher limit.

**(c) How an analyst recognises it.** If growth is logistic, the growth rate per head falls in a straight line with size: $\frac{1}{P}\frac{dP}{dt} = r - \frac{r}{K}P$. So an analyst with yearly counts can regress the growth per head on the level, read $r$ from the intercept and $K$ from where the line crosses zero, using nothing but least squares. We sample 21 yearly values of the S-curve, add 2% noise and fit.

In [ ]:
#| fig-alt: Scatter of the observed growth rate per head against the population for noisy samples of an S-curve, with a fitted straight line falling from about 0.18 at zero to zero near 200.
lg = lb.molecule("logistic")
true = lg.run()
for label, kw in (("base", {}), ("faster engine", {"growth_rate": 35}),
                  ("higher limit", {"carrying_capacity": 350})):
    p = lg.run(**kw)["Population"]
    print(f"{label:<14}: reaches 190 in year {p.index[p >= 190][0]:5.1f}, "
          f"ends at {p.iloc[-1]:.0f}")

# What an analyst would see: yearly counts with 2% noise, years 0 to 20
yearly = true["Population"].iloc[:201:10]
obs = yearly * rng.lognormal(0, 0.02, len(yearly))
growth = np.diff(np.log(obs.values))              # growth per head, per year
size = obs.values[:-1]
slope, intercept = np.polyfit(size, growth, 1)
print(f"\nfitted: r = {intercept:.3f} per year, K = {-intercept / slope:.0f}"
      f" (true r = 0.18, K = 200)")
fig, ax = plt.subplots()
ax.scatter(size, growth, color="#0072B2", label="observed years")
xs = np.linspace(0, 220, 50)
ax.plot(xs, intercept + slope * xs, color="#D55E00", ls="--",
        label="least-squares line")
ax.axhline(0, color="black", lw=0.8)
ax.set(title="Limits to growth in data: growth per head falls with size",
       xlabel="population (units)", ylabel="growth rate (per year)")
ax.legend(fontsize=8)
plt.show()

Figure: Growth per head against population for 20 noisy yearly observations of an S-curve with $r$ = 0.18 a year and $K$ = 200, and the least-squares line.

Doubling the engine (growth rate 35% instead of 18%) gets the population to 190 in year 16.9 instead of 32.8, but it still stops at 200. Raising the limit to 350 is the only change that moves the plateau. From noisy data the fitted line gives $r$ = 0.181 a year and $K$ = 184, close to the truth, and the analyst can say how much room is left before anyone sees the curve bend.

**(b) A real case: water in Cape Town.** For two decades Cape Town's population and water use grew while its dam storage grew far more slowly. The growth engine (a successful city attracting people) ran into a limit (water a drought could take away). After three dry winters from 2015 to 2017 the city came within weeks of a "Day Zero" in 2018 when household taps would be turned off. It was averted by severe restrictions that cut use roughly in half, and the lasting response has been to raise the limit by adding new sources, groundwater, reuse and desalination, rather than to stop the city growing.

**(d) Leverage.** Raise or remove the limit, or slow the engine before it hits it. In the waiting-list case the limit is often hidden: theatre space, beds or anaesthetists. Hiring more consultants (the engine) does nothing once the theatres are full.

**Try it 5.**

- *Apply.* Fit $K$ using only the first 8, 12 and 20 years of the noisy data. How early can the limit be estimated?

<details><summary>Worked answer</summary>

With years 0 to 8, the estimate is 3,392: early growth is almost exponential and says next to nothing about the limit. With years 0 to 12 it is 163, and with 0 to 20 it is 184. Chapter 3 found the same thing by eye; here it is a number. Limits are invisible in the data until growth has started to bend, which is exactly when it is late to act on them.
</details>

In [ ]:
# Apply: fit with only the first 8 years of data
for n in (8, 12, 20):
    o = obs.iloc[:n + 1]
    g, s = np.diff(np.log(o.values)), o.values[:-1]
    b, a = np.polyfit(s, g, 1)
    print(f"years 0 to {n:>2}: K estimate {-a / b:7.0f}")

## 6. Tragedy of the commons

**The pattern.** Several users draw on a shared resource. Each grows its own use while the returns are good, and nobody pays the full cost of depleting the resource. The total use overshoots what the resource can supply, and it collapses for everyone.

**(a) The molecule.** **Tragedy of the Commons** (`tragedyCommons`) is our own model of the archetype [@senge1990], not one of Hines' molecules, though its regrowth is the logistic structure of chapter 3. It has two fleets fishing one stock $N$ that regrows logistically. Each boat catches in proportion to the stock, and each fleet adds boats in proportion to its profit signal, the catch per boat over a break-even catch, minus 1:

$$
\frac{dN}{dt} = rN\Big(1 - \frac{N}{K}\Big) - \text{Catch}, \qquad \text{Catch} = (A + B)\,\frac{c_{\max} N}{K}, \qquad \frac{dA}{dt} = g_A A \Big(\frac{\text{catch per boat}}{\text{break-even}} - 1\Big).
$$

A positive `Catch limit` caps the total catch. Our version uses two identical fleets and a straight-line profit signal so that each fleet's own decision is easy to see.

In [ ]:
#| fig-alt: Two panels over 60 years. Left, fish stock under open access crashes to under 10 and recovers only to the mid 20s; with a catch limit of 8 it stays above 70. Right, catch per boat against the size of both fleets under open access, falling as the fleets grow.
tc = lb.molecule("tragedyCommons")
oa, lim = tc.run(preset="Open access"), tc.run(preset="Catch limit of 8")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
oa["Fish stock"].plot(ax=a1, label="open access")
lim["Fish stock"].plot(ax=a1, label="catch limit of 8")
fleet = oa["Fleet A"] + oa["Fleet B"]
a2.plot(fleet.iloc[:200], oa["Catch per boat"].iloc[:200], color="#0072B2")
a1.set(title="Fish stock", xlabel="time (year)", ylabel="fish stock (kt)")
a2.set(title="Catch per boat as fleets grow (years 0 to 20)",
       xlabel="boats in both fleets", ylabel="catch per boat (kt/year)")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(oa.loc[[0.0, 2.0, 4.0, 8.0, 20.0, 60.0],
             ["Fish stock", "Fleet A", "Fleet B", "Catch",
              "Catch per boat"]].round(2).to_string())
print(f"catch peaks at {oa['Catch'].max():.1f} kt/year in year "
      f"{oa['Catch'].idxmax():.1f}; with a limit of 8 the stock stays above "
      f"{lim['Fish stock'].min():.1f} kt")

Figure: Left, the fish stock under open access and with a catch limit of 8 kt a year. Right, catch per boat against the number of boats in both fleets, years 0 to 20 of open access: the path runs right and down as fleets grow, then back as boats are scrapped.

Under open access both fleets grow fast while catches are good: from 4 boats each to 43.28 each by year 4. The catch peaks at 34.0 kt a year in year 2.6, and by year 8 the stock is down to 7.45 kt. Boats are scrapped, the stock recovers, and the system rings down to a stock of about 26 kt in year 60, where a boat catches 0.26 kt, just above the break-even 0.25. Open access settles where fishing stops paying, not where fish regrow fastest (50 kt). A catch limit of 8 kt a year keeps the stock above 72.4 kt throughout.

**(b) A real case: cod off Newfoundland.** The northern cod off Canada's Atlantic coast supported fishing for centuries. From the 1960s, larger and better-equipped fleets, foreign and then Canadian, took catches far above what the stock could replace, and each party had reasons to keep fishing. In 1992 the stock had collapsed so far that Canada closed the fishery, putting tens of thousands of people out of work. Decades later the stock has recovered only partly.

**(c) How an analyst recognises it.** Total effort rises while the catch per unit of effort falls; the total catch rises, peaks and then falls even as effort keeps growing. The right panel above is that signature: more boats, less fish per boat. In a hospital setting, shared diagnostics such as scanners or theatre lists used by several departments can show the same pattern: more referrals from each department, longer waits for all.

**(d) Leverage.** Manage the resource as a whole: limits on total use, shares that cannot be traded up by racing, or a single owner.

**Try it 6.**

- *Apply.* Regrowth is fastest at half the carrying capacity: $0.4 \times 50 \times (1 - 50/100) = 10$ kt a year. Sweep the catch limit from 6 to 12 kt a year. Which limit lands the most fish over 60 years?
- *Extend.* Run the *One fleet holds back* preset. Does one fleet's restraint save the stock?

<details><summary>Worked answer</summary>

A limit of 10 lands the most, 599 kt over 60 years, and leaves the stock at 53.8 kt. A limit of 11 asks for more than the stock can ever regrow: the stock falls to 22.7 kt and the total catch drops to 566 kt; at 12 it is 539 kt. Overshooting the best limit by 10% costs more than undershooting it by 20% (480 kt at 8). When fleet B stops growing, the stock in year 60 is 27.4 kt, hardly better than under open access: fleet A simply takes the room. Restraint by one user does not save a commons; only a rule for all does.
</details>

In [ ]:
for limit in (6, 8, 10, 11, 12):
    d = tc.run(catch_limit=limit)
    total = d["Catch"].iloc[:-1].sum() * 0.1
    print(f"limit {limit:>2}: total catch {total:4.0f} kt, stock in year 60"
          f" {d['Fish stock'].iloc[-1]:5.1f} kt")
d = tc.run(preset="One fleet holds back")
print("one fleet holds back: stock in year 60",
      round(d["Fish stock"].iloc[-1], 1), "kt, fleet B",
      round(d["Fleet B"].iloc[-1], 1), "boats")

## 7. Success to the successful

**The pattern.** Two parties compete for a resource (customers, funding, talent), and the resource goes preferentially to whoever already has more. An early lead, earned or lucky, compounds into dominance.

**(a) The model.** The Market Share molecule (`marketshare`) from chapter 3 splits new customers by attractiveness that grows with size, $A^{\phi}/(A^{\phi} + B^{\phi})$ with $\phi = 1 + 2 \times \text{network effect}$. Product A starts with 105 customers, B with 95.

**(c) How an analyst recognises it.** Shares drift apart rather than towards each other, and a concentration measure rises. With two parties, the **concentration index** (the sum of squared shares, as in the Herfindahl index competition authorities use) is 0.5 for an even split and 1 for a monopoly.

In [ ]:
#| fig-alt: Concentration index of a two-product market over 60 months for three network effects. With no network effect it stays near 0.5; with a strong one it climbs towards 1.
ms = lb.molecule("marketshare")
fig, ax = plt.subplots()
for label in ("No network effect", "Lock-in", "Winner takes all"):
    d = ms.run(preset=label)
    share = d["Product A"] / (d["Product A"] + d["Product B"])
    hhi = share ** 2 + (1 - share) ** 2
    hhi.plot(ax=ax, label=label)
    print(f"{label:<18}: share of A {share.iloc[0]:.3f} to "
          f"{share.iloc[-1]:.3f}, concentration {hhi.iloc[0]:.3f} to "
          f"{hhi.iloc[-1]:.3f}")
ax.set(title="Success to the successful: concentration over time",
       xlabel="time (month)", ylabel="concentration index (0.5 to 1)")
ax.legend(fontsize=8)
plt.show()

Figure: Concentration index of the two-product market over 60 months, for no network effect, the Lock-in preset (network effect 0.7) and Winner takes all (1).

With no network effect A keeps its 52.5% share and the concentration stays at 0.501. With a network effect of 0.7, A's share grows to 95.9% in five years and concentration to 0.922; with a full network effect, to 97.7% and 0.956. A 10-customer head start became dominance.

**(b) A real case: mobile money in Kenya.** M-Pesa, launched by Safaricom in 2007, let people send money by text message. Every new user made the service more useful to the next, since there were more people to pay and more agents to cash out with, and Safaricom already had the largest mobile network. Within a few years most Kenyan adults used it, and it has held the large majority of the country's mobile money market since, despite competitors. Regulators later pushed for interoperability between services, a way of weakening the link between size and attractiveness.

**(d) Leverage.** Weaken the link between past success and future allocation: interoperability and common standards in markets, or, inside an organisation, budgets set by need rather than by last year's performance. In a hospital group, a unit that meets its targets and so attracts the best staff and the extra funding, while a struggling unit loses both, is the same archetype.

**Try it 7.**

- *Apply.* Start both products at 100 customers with the Winner takes all settings. What happens?
- *Extend.* Now give B the 10-customer lead instead. What does that say about predicting the winner?

<details><summary>Worked answer</summary>

With an equal start the shares stay at exactly 0.5: the structure has no preference, and only an initial difference (or noise) can tip it. With B ahead, A's share ends at 0.023, the mirror image. The outcome depends on small early differences, so in such systems the winner is hard to predict but easy to explain afterwards.
</details>

In [ ]:
# Apply: does a level start stay level?
d = ms.with_init("Product A", 100).with_init("Product B", 100).run(
    preset="Winner takes all")
print("equal start, winner takes all: share of A",
      round(d["Product A"].iloc[-1]
            / (d["Product A"].iloc[-1] + d["Product B"].iloc[-1]), 3))
# Extend: a small lead for B instead
d = ms.with_init("Product A", 95).with_init("Product B", 105).run(
    preset="Winner takes all")
print("B starts 10 ahead: share of A",
      round(d["Product A"].iloc[-1]
            / (d["Product A"].iloc[-1] + d["Product B"].iloc[-1]), 3))

## 8. Eroding goals

**The pattern.** When performance falls short of a goal, the gap can be closed in two ways: improve performance, or lower the goal. Lowering the goal is quick and costs nothing, so under pressure goals drift down, and with them the effort to meet them.

**(a) The model.** In the running case the target list follows the actual list over a goal adjustment time of 24 months, with a weight `Goal_protection` on the original target: the Sea Anchor and Protected Sea Anchor structures of chapter 5, applied to a goal,

$$
\frac{d\,\text{Target}}{dt} = (1 - \text{protection})\,\frac{\text{Waiting list} - \text{Target}}{24}.
$$

**(c) How an analyst recognises it.** Changes in the target follow past gaps between performance and target. So regress each month's change in the target on the previous month's gap: a positive slope means the goal is chasing performance, and its inverse estimates how fast.

In [ ]:
#| fig-alt: Waiting list and target over 120 months when the target follows the list, with no, half and full protection. With no protection both drift up together; with full protection the target stays at 32,000.
fig, ax = plt.subplots()
runs = {}
for gp in (1, 0.5, 0):
    d = case.run(goal_protection=gp)
    runs[gp] = d
    d["Waiting list"].plot(ax=ax, label=f"list, protection {gp}")
    print(f"protection {gp}: target month 120 "
          f"{d['Target list'].iloc[-1]:,.0f}, list month 120 "
          f"{d['Waiting list'].iloc[-1]:,.0f}, capacity "
          f"{d['Public capacity'].iloc[-1]:,.0f}")
runs[0]["Target list"].plot(ax=ax, label="target, protection 0",
                            color="grey")
ax.set(title="Eroding goals: a target that follows the list",
       ylabel="patients")
month_axis(ax)
ax.legend(fontsize=8)
plt.show()

# What an analyst would see: does the target move towards past results?
d = runs[0].iloc[::8]                             # one row a month
tgt = d["Target list"] * rng.lognormal(0, 0.002, len(d))
change = np.diff(tgt.values)                      # patients per month
gap = (d["Waiting list"] - tgt).values[:-1]       # patients
b, a = np.polyfit(gap, change, 1)
print(f"\ntarget change = {a:.1f} + {b:.4f} x gap: the target closes "
      f"{b:.1%} of the gap each month, an adjustment time of "
      f"{1 / b:.0f} months")

Figure: The waiting list with full, half and no protection of the target, and the target itself when it is not protected. Running case, January 2014 to December 2023.

With the target fixed, the list is back to 32,169 by month 120. With half protection the target drifts to 33,593 and the list settles at 33,677; with none the target drifts to 34,685 and the list to 34,715. Capacity barely differs (4,386 to 4,399 patients a month): the hospital is not working any less hard, it simply stops trying to recover the ground lost after the referral rise. The regression on the unprotected run finds that the target closes 4.7% of the gap each month, an adjustment time of 21 months, near the true 24 (the difference comes from the noise and from fitting one month at a time).

**(b) A real case: the four-hour standard in English emergency departments.** In 2004 the NHS in England set a standard that 98% of patients in emergency departments should be seen and admitted or discharged within four hours. In 2010 it was relaxed to 95%. After years of missing it, a 2023 recovery plan set an interim aim of 76% by March 2024. Each step had good reasons, and each made the next shortfall look less bad.

**(d) Leverage.** Anchor the goal to something outside the system's own results: a clinical standard, the best comparable performer, or the patients' view of an acceptable wait. If a target must change, make the change explicit and justified, not the silent result of repeated misses.

## 9. The running case: three quick fixes at once

Real managers do not apply one fix at a time. We now switch on all three quick fixes together (overtime up to 1.3, rushing up to 1.3, 90% of the excess list outsourced with a moderate staff drift of 0.05) with a half-protected target. Then we apply the leverage from each section: overtime limited to 1.1, no rushing, outsourcing with a three-year sunset and capacity planned as if outsourcing will stop, and a fixed target. The scorecard shows what a board would see.

In [ ]:
#| fig-alt: "Waiting list over 120 months for three policies: no quick fixes; all three quick fixes; and the fixes with the leverage applied. All quick fixes make the list lower at first and higher later."
plan_all = case.with_equation(
    "Building capacity",
    "MAX(0, Public_capacity / Capacity_life + (Desired_treatments"
    " - Public_capacity) / Time_to_build_capacity)")
quick = dict(overtime_limit=1.3, rush_limit=1.3, outsourcing_share=0.9,
             staff_drift=0.05, goal_protection=0.5)
lever = dict(overtime_limit=1.1, rush_limit=1, outsourcing_share=0.9,
             staff_drift=0.05, goal_protection=1, outsourcing_end=36)
policies = {"no quick fixes": (case, {}),
            "all quick fixes": (case, quick),
            "fixes with leverage": (plan_all, lever)}


def scorecard(d, dt=0.125, target=32000):
    """Board-level measures of one run."""
    lst = d["Waiting list"]
    return pd.Series({
        "list month 24": lst.loc[24.0],
        "list month 60": lst.loc[60.0],
        "list month 120": lst.iloc[-1],
        "patient-months above 32,000": (lst - target).clip(lower=0).sum() * dt,
        "returns (patients)": d["Returning"].sum() * dt,
        "outsourced (patients)": d["Outsourced treatments"].sum() * dt,
        "capacity month 120": d["Public capacity"].iloc[-1]})


fig, ax = plt.subplots()
table = {}
for label, (m, kw) in policies.items():
    d = m.run(**kw)
    d["Waiting list"].plot(ax=ax, label=label)
    table[label] = scorecard(d)
ax.axhline(32000, color="grey", lw=1, ls=":")
ax.set(title="Three quick fixes together, and with leverage",
       ylabel="waiting list (patients)")
month_axis(ax)
ax.legend(fontsize=8)
plt.show()
print(pd.DataFrame(table).round(0).astype(int).to_string())

Figure: The running case under three policies: no quick fixes; all three quick fixes with a half-protected target; and the fixes with the leverage applied.

The quick fixes do what they promise at first: the list in month 24 is 34,765 against 35,560 without them. By month 60 it is 37,168 against 32,169, and in month 120 it is 38,947, still rising, with public capacity 104 patients a month lower than without the fixes. Over ten years the fixes add up to 541,238 patient-months above the original target, more than four times the 121,524 of doing nothing quick at all, along with 8,864 returning patients and 18,787 outsourced treatments. The archetypes compound: fatigue and rework add demand, outsourcing removes the pressure to build, and the drifting target hides the result.

With the leverage applied, the list in month 24 is the lowest of the three (34,507), it ends at 31,610, below target, and the cost is 149,163 patient-months above target, 23% more than with no quick fixes at all, but with a better position in month 24 and month 120. Quick fixes are not the problem; quick fixes without attention to their side effects are.

**Try it 9.**

- *Check.* In the leverage policy, how many patients are outsourced in total, and why so few?
- *Apply.* Starting from the leverage policy, put the overtime limit back to 1.3. What happens to returns and to the list in month 60, and why?

<details><summary>Worked answer</summary>

3,854 patients are outsourced under the leverage policy, against 18,787 under the quick-fix policy, mainly because the sunset stops outsourcing after month 36. Putting the overtime limit back to 1.3 changes almost nothing: returns go from 4,342 to 4,351 and the list in month 60 from 34,558 to 34,559. With 90% of the excess list outsourced in the early years, the pressure on public clinics rarely calls for more than about 10% overtime, so a limit of 1.1 hardly binds. A cap that never binds is not a policy; the analyst should check which constraints are active before recommending one.
</details>

In [ ]:
# Check: patients outsourced under each policy
print(pd.DataFrame(table).loc["outsourced (patients)"].round(0).to_string())
# Apply: the leverage policy with the overtime limit back at 1.3
more = scorecard(plan_all.run(**dict(lever, overtime_limit=1.3)))
print(pd.DataFrame({"leverage": table["fixes with leverage"],
                    "leverage, overtime 1.3": more}).round(0)
      .astype(int).to_string())

## Check it

Four checks for a model with several fixes: units on every element; a balanced start with every fix switched on must stay put (no fix should act when there is no gap); patients must be conserved through the hidden returns; and halving `dt` must not change the result.

In [ ]:
# 1. Units on every element
print("missing units:", [e["name"] for e in case.stocks + case.flows
                         + case.vars if not e.get("units")])
# 2. Extreme conditions: balanced start, no rise, every fix switched on
calm = case.with_init("Public capacity", 4000).run(
    referral_rise=0, overtime_limit=1.3, rush_limit=1.3,
    outsourcing_share=0.9, goal_protection=0)
print("balanced start, fixes on: list", round(calm["Waiting list"].min(), 1),
      "to", round(calm["Waiting list"].max(), 1), "| overtime max",
      calm["Overtime"].max())
# 3. Conservation: patients in = patients out + change in stocks
d = case.run(**quick)
dt = 0.125
inflow = d["Referrals"].iloc[:-1].sum() * dt
outflow = (d["Treated and discharged"] + d["Outsourced treatments"]
           ).iloc[:-1].sum() * dt
stocks = (d["Waiting list"] + d["Hidden returns"])
print(f"in {inflow:,.1f}, out {outflow:,.1f}, change in stocks "
      f"{stocks.iloc[-1] - stocks.iloc[0]:,.1f}, gap "
      f"{inflow - outflow - (stocks.iloc[-1] - stocks.iloc[0]):.4f}")
# 4. Halve dt
for step in (0.25, 0.125, 0.0625):
    d = case.run(dt=step, **quick)
    print(f"dt {step:<6}: list month 120 {d['Waiting list'].iloc[-1]:9.1f}")

Every element has units. Started with capacity equal to referrals and no rise, the list stays at exactly 32,000 with every fix switched on and overtime never rises above 1.0: the fixes respond only to a gap. Over the quick-fix run, 523,200 patients are referred, 516,029.2 leave (treated and discharged, or outsourced) and the waiting list and hidden returns together grow by 7,170.8: the books balance to four decimal places. Halving the time step from 0.25 to 0.125 to 0.0625 month moves the list in month 120 from 38,948.0 to 38,946.6 to 38,945.9 patients, well below anything we report.

## Running case

This chapter added the quick fixes managers use while capacity is built:

| Element | What it is | Value |
|---|---|---|
| `Public capacity` (stock) | patients a month the public system can treat | starts at 3,950; life 120 months; built over 18 months |
| `Fatigue` (stock) | smooth of overtime | 3 months; strength 0.8 on productivity |
| `Hidden returns` (stock) | patients treated with flaws, not yet back | return after 4 months |
| `Overtime_limit`, `Rush_limit` | how far staff stretch | 1 (off); 1.3 in the scenarios |
| `Outsourcing_share`, `Staff_drift` | share of the excess list bought privately; drain on public capacity | 0 (off); 0.9 and 0 to 0.1 in the scenarios |
| `Goal_protection` | weight on the original target | 1 (fixed); 0 and 0.5 in the scenarios |

Part III starts with chapter 8, Learning from data, which gives the statistics and machine learning the rest of the book needs. Chapter 9, Testing your model, then puts the waiting-list model through the tests that build confidence in it before it is calibrated to data in chapter 10.

## Tell the decision maker

- **Every quick fix in our case helps for about two years and then costs more than it saved.** Together, overtime, rushed clinics and open-ended outsourcing leave the list about 6,800 patients longer after ten years than doing none of them, and still rising.
- **Outsourcing is a bridge or a crutch depending on one number**: how much public capacity it draws away. Measure it. Above a threshold it hollows out the public system; a sunset clause and planning public capacity as if outsourcing will stop both keep it a bridge.
- **Count returning patients separately from new referrals.** Rushed clinics showed up as returns equal to 9% of referrals, two years after the decision to rush.
- **Fix the target in advance.** A target that follows performance hides the problem: in our run it drifted up by 2,700 patients with no change in effort.

## Going further

- Archetypes can be identified more formally by loop dominance analysis: which loop drives a stock's behaviour at each moment. Methods based on eigenvalues and on pathway participation exist, and the simplest version (comparing the size of each loop's flows over time) needs only the run's DataFrame.
- The threshold in section 4 was found by algebra on the equilibrium and confirmed by simulation. For larger models, a sweep with `run_batch` over the uncertain side-effect strength, as in chapter 6, does the same job, and chapter 10 shows how to put a probability on which side of the threshold a real system lies.
- The fuller catalogue of archetypes includes escalation (two parties each responding to the other's threat, as in price wars and arms races), growth and underinvestment, and accidental adversaries; each can be built from the molecules in this book.

## Mini-project

Build **escalation** into the running case. Two hospital groups in the same region recruit from one pool of agency nurses, and each raises its pay offer whenever its vacancies exceed the other's (use two stocks of `Pay offer` that adjust towards a mark-up on the other's offer). Show the escalation in pay, find a leverage point (a shared regional pay rate, or a cap), and report the cost in a scorecard like the one in section 9. Then say, in a paragraph for the regional health authority, which archetype from this chapter your result most resembles and why.

## Further reading

- @senge1990 introduced the system archetypes to a management audience, with many organisational examples.
- @meadows2008 describes the same traps as "system traps" and pairs each with its way out.
- @sterman2000 covers fixes that fail, rework and the capability trap with formal models.
- Hines, *Molecules of Structure* [@hines_molecules], describes the overtime, fatigue, quality, rework and schedule pressure structures used here.

If you would like to play with one of these structures, the LoopBricks app has an interactive version of [Shifting the Burden](https://waelrash1.github.io/loopbricks/#/m/shiftingBurden).